In [1]:
import os

from denovo_utils.data import Run
from denovo_utils.parsers import DenovoEngineConverter
from denovo_utils.io.read import load_psmlist

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

from denovo_utils.analysis.metrics import load_seq_score_dicts
from denovo_utils.analysis.metrics import get_match_score_table, get_prc_curve

from psm_utils import Peptidoform
from tqdm import tqdm
from peak_pack.utils import calculate_ppm

from denovo_utils.analysis.metrics import (
    get_refinement_error_tables,
    plot_refinement_error_table,
    plot_precision_recall_refinement
)

import pandas as pd

from pathlib import Path

from denovo_utils.io.read import read_partitions_features, read_partitions_psmlist
from denovo_utils.io.save import save_psmlist

from glob import glob

2026-02-25 13:39:25.350936: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Modification already exists in ModificationsDB. Skipping.


# Build the Run objects used for following notebooks

In [2]:
root_rescoring = '/public/compomics3/Sam/PXD028735/QExactive/reviewer/rescoring/chimeric'

run_names = [
    'LFQ_Orbitrap_DDA_Ecoli_01',
    'LFQ_Orbitrap_DDA_Human_01',
    'LFQ_Orbitrap_DDA_QC_01',
    'LFQ_Orbitrap_DDA_Yeast_01'
]

denovo_names = [
    'adanovo',
    'pihelixnovo',
    'piprimenovo',
    'casanovo',
    'instanovo',
    'contranovo',
    'novob',
    'pepnet',
]

refinement_names = [
    'spectralis',
    'instanovoplus'
]

In [3]:
runs = {}
for run_name in run_names:
    print(f"Run: {run_name}")
    run = Run(run_name)

    gt_path = os.path.join(root_rescoring, run_name, 'psmlist', 'sage_msgf.parquet')
    psmlist_gt = load_psmlist(gt_path)
    run.load_gold_standard(
        psmlist=psmlist_gt,
        score_name='score_ms2rescore',
        filter_decoys=True,
        filter_on_qvalue=True
    )

    for denovo_name in denovo_names:
        print(f'loading {denovo_name}')

        denovo_path = os.path.join(root_rescoring, run_name, 'psmlist', f'{denovo_name}.parquet')
        psmlist_denovo = load_psmlist(denovo_path)
        run.load_psmlist(
            psmlist=psmlist_denovo,
            score_names=['score_ms2rescore'],
            only_gold_standard_spectra=False
        )
    
        # load refinement
        for refinement_name in refinement_names:
            refinement_path = os.path.join(root_rescoring, run_name, 'psmlist', f'{denovo_name}.{refinement_name}.parquet')
            if os.path.exists(refinement_path):
                print(f'    loading {refinement_name}')
                psmlist_refinement = load_psmlist(refinement_path)
                run.load_psmlist_refinement(psmlist_refinement, overwrite=True)

    runs[run_name] = run

Run: LFQ_Orbitrap_DDA_Ecoli_01


Loading results from sage_msgf.parquet: 100%|██████████| 27815/27815 [00:02<00:00, 10855.68it/s]


Q-value (True) filtering: 0 PSMs filtered.
Decoy filtering: 0 PSMs filtered.


Loading PSMs in Run object (LFQ_Orbitrap_DDA_Ecoli_01): 100%|██████████| 27815/27815 [00:00<00:00, 203544.11it/s]


loading adanovo


...

Loading results from pepnet.parquet: 100%|██████████| 103785/103785 [00:07<00:00, 14440.53it/s]
Loading PSMs in Run object (LFQ_Orbitrap_DDA_Yeast_01): 100%|██████████| 103785/103785 [00:00<00:00, 131750.72it/s]


    loading spectralis


Loading results from pepnet.spectralis.parquet: 100%|██████████| 103785/103785 [00:06<00:00, 15717.62it/s]


    loading instanovoplus


Loading results from pepnet.instanovoplus.parquet: 100%|██████████| 103769/103769 [00:06<00:00, 16596.97it/s]


In [4]:
import pickle

In [5]:
with open('runs_without_evaluation.pkl', 'wb') as f:
    pickle.dump(runs, f)

## 2. Compare to ground-truth

In [7]:
# Perform error type analysis
for run in runs.values():
    for spectrum in tqdm(run.spectra.values()):
        spectrum.compare_gold_standard(
            metadata_score='score_ms2rescore',
            refinements=[
                'InstaNovo+',
                'Spectralis'
            ],
        )

100%|██████████| 103785/103785 [01:40<00:00, 1033.29it/s]


In [12]:
def get_run_with_GS(run):
    run.spectra = {
        specid: spectrum for specid, spectrum in run.spectra.items() if len(spectrum.psm_gold_standard)>0
    }
    return run

runs = {
    run_name: get_run_with_GS(run) for run_name, run in runs.items()
}

In [21]:
runs['LFQ_Orbitrap_DDA_Ecoli_01'].spectra['controllerType=0 controllerNumber=1 scan=10005'].psm_candidates[
    0
].refinement['Spectralis'][0].evaluations

{1: {'score_ms2rescore': ('match', -0.0)}}

In [24]:
runs['LFQ_Orbitrap_DDA_Ecoli_01'].spectra['controllerType=0 controllerNumber=1 scan=10005'].psm_candidates[
    0
].refinement['Spectralis'][0]

{'peptidoform': Peptidoform('VIC[UNIMOD:4]SAEPK/2'), 'scores': {'peptide': {'Spectralis': -0.33040812611579895, 'score_ms2rescore': -0.03617122288384156}, 'aa': {}}, 'engine_name': 'Spectralis', 'peptide_evidence': VIC[UNIMOD:4]SAEPK/2, 'refinement': {}, 'evaluation': {1: {'score_ms2rescore': ('match', -0.0)}}}

In [30]:
from denovo_utils.parsers.io.fasta import FastaHandler

def in_fasta(sequence: str, fasta, fasta_type='str'):
    sequence = sequence.replace('L', 'I')
    if fasta_type=='str':
        return sequence in fasta
    else:
        for fasta_sequence in fasta:
            if sequence in fasta_sequence:
                return True
        return False
    
def has_good_eval(psm_candidate, good_labels=['match']):
    for evaluation in psm_candidate.evaluations.values():
        if evaluation['score_ms2rescore'].error_type in good_labels:
            return True
    return False

def rannotate_error_type(run: Run, good_labels, fasta_str):
    for spectrum in tqdm(run.spectra.values(), desc=f'Reannotating error types for run {run.run_id}'):
        
        for psm in spectrum.psm_candidates:
            good_eval = has_good_eval(psm, good_labels)
            
            if not good_eval:
                    
                if in_fasta(psm.peptidoform.sequence, fasta=fasta_str):
                    psm.metadata['match_type'] = 'No match - In FASTA'
                else:
                    psm.metadata['match_type'] = 'No match - Not in FASTA'
            
            else:
                psm.metadata['match_type'] = 'match'

            # Check refinement sequences
            for refinement_name, refinement in psm.refinement.items():
                if not refinement[-1]: # True when same PSM as base
                    psm_refinement = refinement[0]
                    good_eval = has_good_eval(psm_refinement)
                    if not good_eval:
                        if in_fasta(psm_refinement.peptidoform.sequence, fasta=fasta_str):
                            psm_refinement.metadata['match_type'] = 'No match - In FASTA'
                        else:
                            psm_refinement.metadata['match_type'] = 'No match - Not in FASTA'
                    else:
                        psm_refinement.metadata['match_type'] = 'match'

In [31]:
fh = FastaHandler()
fh.read(path_fasta='/public/compomics3/Sam/denovo_paper/fasta/human_ecoli_yeast_contaminant_revi_2025_10_17.fasta')
fasta_sequences = fh.dataframe['sequence'].tolist()
fasta_seq_concat = " ".join([str(i).replace('L', 'I') for i in fasta_sequences])

In [33]:
# Reannnotate the error types
for run in runs.values():
    rannotate_error_type(
        run=run,
        good_labels=['match'],
        fasta_str=fasta_seq_concat
    )

Reannotating error types for run LFQ_Orbitrap_DDA_Ecoli_01: 100%|██████████| 19820/19820 [12:50<00:00, 25.73it/s]
Reannotating error types for run LFQ_Orbitrap_DDA_Human_01: 100%|██████████| 72028/72028 [37:18<00:00, 32.17it/s]
Reannotating error types for run LFQ_Orbitrap_DDA_QC_01: 100%|██████████| 67043/67043 [36:53<00:00, 30.29it/s]
Reannotating error types for run LFQ_Orbitrap_DDA_Yeast_01: 100%|██████████| 49588/49588 [28:00<00:00, 29.51it/s]


In [34]:
with open('runs_PXD028735_QExactive.pkl', 'wb') as f:
    pickle.dump(runs, f)

## 3. Create sequence comparisons

In [35]:
from denovo_utils.analysis.error_types import build_comparison_table
comparison_tables = {}

for denovo_name, run in runs.items():
    comparison_tables[denovo_name] = build_comparison_table(
        run=run
    )

Calculating sequence distance for de novo PSMs: 100%|██████████| 49588/49588 [02:38<00:00, 313.01it/s]


In [36]:
with open('comparison_tables_PXD028735.pkl', 'wb') as f:
    pickle.dump(comparison_tables, f)

# 4. Populate timsTOF run object

In [51]:
run_names = [
    'LFQ_timsTOFPro_PASEF_Ecoli_01',
    'LFQ_timsTOFPro_PASEF_Human_01',
    'LFQ_timsTOFPro_PASEF_QC_01',
    'LFQ_timsTOFPro_PASEF_Yeast_01'
]
root_mgf = '/public/compomics3/Sam/PXD028735/timsTOF/mgf_reformatted'
root_denovo_output = '/public/compomics3/Sam/PXD028735/timsTOF/denovo_output'
root_ground_truth = '/public/compomics3/Sam/PXD028735/timsTOF/reviewer/rescoring'
root_refinement = ''
format_ground_truth = 'parquet'
root_rescoring = '/public/compomics3/Sam/PXD028735/timsTOF/reviewer/rescoring'

runs = {}
for run_name in run_names:
    print(f"Run: {run_name}")
    run = Run(run_name)

    gt_path = os.path.join(root_rescoring, run_name, 'psmlist', 'sage_msgf.parquet')
    psmlist_gt = load_psmlist(gt_path)
    run.load_gold_standard(
        psmlist=psmlist_gt,
        score_name='score_ms2rescore',
        filter_decoys=True,
        filter_on_qvalue=True
    )

    for denovo_name in denovo_names:
        print(f'loading {denovo_name}')

        path_denovo_file = os.path.join(
            root_denovo_output,
            denovo_name,
            run_name + '.' + denovo_name + '.some_extension'
        )
        mgf_path = os.path.join(
            root_mgf, run_name+'.mgf'
        )
        parser = DenovoEngineConverter.select(denovo_name.replace('instanovo', 'instanovo_v11'))
        psmlist_denovo = parser.parse(
            result_path=path_denovo_file,
            mgf_path=mgf_path
        )
        run.load_psmlist(
            psmlist=psmlist_denovo,
            score_names=[],
            only_gold_standard_spectra=False
        )

    runs[run_name] = run

Run: LFQ_timsTOFPro_PASEF_Ecoli_01
Q-value (True) filtering: 0 PSMs filtered.
Decoy filtering: 0 PSMs filtered.


Loading PSMs in Run object (LFQ_timsTOFPro_PASEF_Ecoli_01): 100%|██████████| 98685/98685 [00:00<00:00, 201532.98it/s]


loading adanovo


Parsing Casanovo results to PSMList: 100%|██████████| 48669/48669 [00:01<00:00, 33089.65it/s]
Loading PSMs in Run object (LFQ_timsTOFPro_PASEF_Ecoli_01): 100%|██████████| 48669/48669 [00:01<00:00, 46534.73it/s]


loading pihelixnovo


...

Loading PSMs in Run object (LFQ_timsTOFPro_PASEF_Yeast_01): 0it [00:00, ?it/s]


loading novob


Parsing NovoB results to PSMList: 100%|██████████| 42880/42880 [00:01<00:00, 22863.60it/s]
Loading PSMs in Run object (LFQ_timsTOFPro_PASEF_Yeast_01): 100%|██████████| 42174/42174 [00:00<00:00, 181677.97it/s]


loading pepnet


Parsing PepNet results to PSMList: 100%|██████████| 67459/67459 [00:01<00:00, 42790.30it/s]
Loading PSMs in Run object (LFQ_timsTOFPro_PASEF_Yeast_01): 100%|██████████| 67459/67459 [00:01<00:00, 41192.63it/s]


In [45]:


# runs_timstof = {}
# for run_name in run_names:
#     print(f"Run: {run_name}")
#     run = Run(run_name)
    
#     gt_path = os.path.join(root_rescoring, run_name, 'psmlist', 'sage_msgf.parquet')
#     psmlist_gt = load_psmlist(gt_path)
#     run.load_gold_standard(
#         psmlist=psmlist_gt,
#         score_name='score_ms2rescore',
#         filter_decoys=True,
#         filter_on_qvalue=True
#     )

#     for denovo_name in denovo_names:
#         print(f'loading {denovo_name}')

#         path_denovo_file = os.path.join(
#             root_denovo_output,
#             denovo_name,
#             run_name + '.' + denovo_name + '.some_extension'
#         )
#         parser = DenovoEngineConverter.select(denovo_name.replace('instanovo', 'instanovo_v11'))
#         psmlist_denovo = parser.parse(
#             result_path=path_denovo_file,
#             mgf_path=mgf_path
#         )
#         run.load_psmlist(
#             psmlist=psmlist_denovo,
#             score_names=[],
#             only_gold_standard_spectra=False
#         )
#     runs_timstof[run_name] = run

KeyboardInterrupt: 

In [54]:
with open('runs_PXD028735_timsTOF.pkl', 'wb') as f:
    pickle.dump(runs, f)

# 5. Populate metaproteomics run object

In [49]:
root_denovo_output = '/public/compomics3/Sam/PXD023217/denovo_output'
root_mgf = '/public/compomics3/Sam/PXD023217/mgf_filtered'
root_ground_truth = '/public/compomics3/Sam/PXD023217/reviewer/rescoring'

# The other files should be rerun with ContraNovo due to bad spectrum_id parsing
run_names = [
    'S03',
    'S05',
    'S07',
]

engine_names = [
    'adanovo',
    'pihelixnovo',
    'piprimenovo',
    'casanovo',
    'instanovo',
    'contranovo',
    'novob',
    'pepnet',
]

root_refinement = ''

format_ground_truth = 'parquet'

runs_metaproteomics = {}
for run_name in run_names:
    print(f"Run: {run_name}")
    run = Run(run_name)
    
    mgf_path = os.path.join(root_mgf, run_name+'.mgf')
    gt_path = os.path.join(root_ground_truth, run_name, 'psmlist', 'sage_msgf.parquet')
    psmlist_gt = load_psmlist(gt_path)
    run.load_gold_standard(
        psmlist=psmlist_gt,
        score_name='score_ms2rescore',
        filter_decoys=True,
        filter_on_qvalue=True
    )

    for denovo_name in denovo_names:
        print(f'loading {denovo_name}')

        path_denovo_file = os.path.join(
            root_denovo_output,
            denovo_name,
            run_name + '.' + denovo_name + '.some_extension'
        )
        parser = DenovoEngineConverter.select(denovo_name.replace('instanovo', 'instanovo_v11'))
        psmlist_denovo = parser.parse(
            result_path=path_denovo_file,
            mgf_path=mgf_path
        )
        run.load_psmlist(
            psmlist=psmlist_denovo,
            score_names=[],
            only_gold_standard_spectra=False
        )
    runs_metaproteomics[run_name] = run

Run: S03


Loading results from sage_msgf.parquet: 100%|██████████| 168295/168295 [00:13<00:00, 12158.89it/s]


Q-value (True) filtering: 0 PSMs filtered.
Decoy filtering: 0 PSMs filtered.


Loading PSMs in Run object (S03): 100%|██████████| 168295/168295 [00:00<00:00, 173713.78it/s]


loading adanovo


...

Loading PSMs in Run object (S03): 100%|██████████| 131605/131605 [00:00<00:00, 207906.44it/s]


loading pepnet


Loading PSMs in Run object (S03): 100%|██████████| 152378/152378 [00:03<00:00, 42590.52it/s]

Run: S05


FileNotFoundError: [Errno 2] Failed to open local file '/public/compomics3/Sam/PXD023217/reviewer/rescoring/S05/psmlist/sage_msgf.parquet'. Detail: [errno 2] No such file or directory

In [44]:
with open('runs_PXD023217.pkl', 'wb') as f:
    pickle.dump(runs_metaproteomics, f)